# Regularization: Ridge, Lasso, and Elastic Net

## Overview
Regularization techniques prevent overfitting by adding a penalty term to the cost function. This constrains the model's coefficients, leading to simpler, more generalizable models.

## Learning Objectives
- Understand the problem of overfitting and how regularization helps
- Learn Ridge (L2), Lasso (L1), and Elastic Net regularization
- Implement regularized regression from scratch and with scikit-learn
- Choose appropriate regularization strength (hyperparameter tuning)
- Understand feature selection with Lasso

## Mathematical Foundation

### Linear Regression Cost Function (OLS)
$$J(\boldsymbol{\beta}) = \frac{1}{2m}\sum_{i=1}^{m}(h_\boldsymbol{\beta}(\mathbf{x}^{(i)}) - y^{(i)})^2$$

### Ridge Regression (L2 Regularization)
$$J(\boldsymbol{\beta}) = \frac{1}{2m}\sum_{i=1}^{m}(h_\boldsymbol{\beta}(\mathbf{x}^{(i)}) - y^{(i)})^2 + \alpha\sum_{j=1}^{n}\beta_j^2$$

Closed form:
$$\boldsymbol{\beta} = (\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

### Lasso Regression (L1 Regularization)
$$J(\boldsymbol{\beta}) = \frac{1}{2m}\sum_{i=1}^{m}(h_\boldsymbol{\beta}(\mathbf{x}^{(i)}) - y^{(i)})^2 + \alpha\sum_{j=1}^{n}|\beta_j|$$

### Elastic Net (L1 + L2)
$$J(\boldsymbol{\beta}) = \frac{1}{2m}\sum_{i=1}^{m}(h_\boldsymbol{\beta}(\mathbf{x}^{(i)}) - y^{(i)})^2 + \alpha\rho\sum_{j=1}^{n}|\beta_j| + \frac{\alpha(1-\rho)}{2}\sum_{j=1}^{n}\beta_j^2$$

Where $\alpha$ is the regularization strength and $\rho$ is the L1 ratio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Demonstrating the Need for Regularization

Let's create a scenario where overfitting occurs.

In [ ]:
# Generate data with noise
np.random.seed(42)
m = 20  # Small dataset
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.randn(m, 1) * 1.5

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_seed=42)

# Create high-degree polynomial features (prone to overfitting)
degree = 15
poly_features = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Standard linear regression (no regularization)
lin_reg = LinearRegression()
lin_reg.fit(X_train_poly, y_train)

# Predictions
y_train_pred = lin_reg.predict(X_train_poly)
y_test_pred = lin_reg.predict(X_test_poly)

print(f"Dataset: {m} samples, polynomial degree {degree}")
print(f"Number of features: {X_train_poly.shape[1]}")
print(f"\nLinear Regression (No Regularization):")
print(f"  Training R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"  Test R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"  Training RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"\n⚠️ Overfitting detected: Perfect training fit but poor test performance!")

## 2. Ridge Regression (L2 Regularization)

Ridge adds the squared magnitude of coefficients as penalty term.

In [ ]:
# Ridge regression with different alpha values
alphas = [0, 0.01, 0.1, 1, 10, 100]
results_ridge = []

for alpha in alphas:
    ridge_reg = Ridge(alpha=alpha)
    ridge_reg.fit(X_train_poly, y_train)
    
    y_train_pred = ridge_reg.predict(X_train_poly)
    y_test_pred = ridge_reg.predict(X_test_poly)
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    results_ridge.append({
        'alpha': alpha,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse
    })

df_ridge = pd.DataFrame(results_ridge)
print("Ridge Regression Results:")
print(df_ridge.to_string(index=False))

In [ ]:
# Visualize Ridge performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² scores
axes[0].plot(df_ridge['alpha'], df_ridge['train_r2'], 'bo-', label='Training R²', linewidth=2)
axes[0].plot(df_ridge['alpha'], df_ridge['test_r2'], 'ro-', label='Test R²', linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (Regularization Strength)', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('Ridge Regression: R² vs Alpha', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# RMSE
axes[1].plot(df_ridge['alpha'], df_ridge['train_rmse'], 'bo-', label='Training RMSE', linewidth=2)
axes[1].plot(df_ridge['alpha'], df_ridge['test_rmse'], 'ro-', label='Test RMSE', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (Regularization Strength)', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('Ridge Regression: RMSE vs Alpha', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_alpha_ridge = df_ridge.loc[df_ridge['test_r2'].idxmax(), 'alpha']
print(f"\nOptimal alpha for Ridge: {best_alpha_ridge}")

## 3. Lasso Regression (L1 Regularization)

Lasso can drive some coefficients exactly to zero, performing feature selection.

In [ ]:
# Lasso regression with different alpha values
results_lasso = []

for alpha in alphas:
    lasso_reg = Lasso(alpha=alpha, max_iter=10000)
    lasso_reg.fit(X_train_poly, y_train)
    
    y_train_pred = lasso_reg.predict(X_train_poly)
    y_test_pred = lasso_reg.predict(X_test_poly)
    
    # Count non-zero coefficients
    n_nonzero = np.sum(lasso_reg.coef_ != 0)
    
    results_lasso.append({
        'alpha': alpha,
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'n_features': n_nonzero
    })

df_lasso = pd.DataFrame(results_lasso)
print("Lasso Regression Results:")
print(df_lasso.to_string(index=False))

In [ ]:
# Visualize Lasso performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R² scores
axes[0].plot(df_lasso['alpha'], df_lasso['train_r2'], 'bo-', label='Training R²', linewidth=2)
axes[0].plot(df_lasso['alpha'], df_lasso['test_r2'], 'ro-', label='Test R²', linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('Lasso: R² vs Alpha', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# RMSE
axes[1].plot(df_lasso['alpha'], df_lasso['train_rmse'], 'bo-', label='Training RMSE', linewidth=2)
axes[1].plot(df_lasso['alpha'], df_lasso['test_rmse'], 'ro-', label='Test RMSE', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('Lasso: RMSE vs Alpha', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Number of features
axes[2].plot(df_lasso['alpha'], df_lasso['n_features'], 'go-', linewidth=2, markersize=8)
axes[2].set_xscale('log')
axes[2].set_xlabel('Alpha', fontsize=12)
axes[2].set_ylabel('Number of Non-Zero Coefficients', fontsize=12)
axes[2].set_title('Lasso: Feature Selection', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLasso performs automatic feature selection!")
print(f"At alpha=0.1: {df_lasso[df_lasso['alpha']==0.1]['n_features'].values[0]} features selected")

## 4. Comparing Ridge vs Lasso: Coefficient Paths

In [ ]:
# Coefficient paths for Ridge and Lasso
alphas_path = np.logspace(-3, 3, 50)
ridge_coefs = []
lasso_coefs = []

for alpha in alphas_path:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly, y_train)
    ridge_coefs.append(ridge.coef_.ravel())
    
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_poly, y_train)
    lasso_coefs.append(lasso.coef_.ravel())

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

# Plot coefficient paths
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ridge coefficients
for i in range(ridge_coefs.shape[1]):
    axes[0].plot(alphas_path, ridge_coefs[:, i], alpha=0.7)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('Ridge: Coefficient Paths', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', linewidth=1)

# Lasso coefficients
for i in range(lasso_coefs.shape[1]):
    axes[1].plot(alphas_path, lasso_coefs[:, i], alpha=0.7)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('Coefficient Value', fontsize=12)
axes[1].set_title('Lasso: Coefficient Paths', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("  Ridge: Coefficients shrink smoothly but never reach exactly zero")
print("  Lasso: Coefficients can become exactly zero (sparse solutions)")

## 5. Elastic Net: Best of Both Worlds

In [ ]:
# Elastic Net with different l1_ratio values
l1_ratios = [0, 0.25, 0.5, 0.75, 1.0]  # 0=Ridge, 1=Lasso
alpha = 0.1

results_elastic = []
for l1_ratio in l1_ratios:
    elastic = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    elastic.fit(X_train_poly, y_train)
    
    y_test_pred = elastic.predict(X_test_poly)
    n_nonzero = np.sum(elastic.coef_ != 0)
    
    results_elastic.append({
        'l1_ratio': l1_ratio,
        'type': 'Ridge' if l1_ratio == 0 else ('Lasso' if l1_ratio == 1 else 'Elastic Net'),
        'test_r2': r2_score(y_test, y_test_pred),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'n_features': n_nonzero
    })

df_elastic = pd.DataFrame(results_elastic)
print(f"Elastic Net Results (alpha={alpha}):")
print(df_elastic.to_string(index=False))
print(f"\nl1_ratio=0: Pure Ridge (L2)")
print(f"l1_ratio=1: Pure Lasso (L1)")
print(f"l1_ratio∈(0,1): Combination of L1 and L2")

## 6. Automatic Hyperparameter Tuning with Cross-Validation

In [ ]:
# RidgeCV: Automatic alpha selection using cross-validation
alphas_cv = np.logspace(-3, 3, 50)
ridge_cv = RidgeCV(alphas=alphas_cv, cv=5)
ridge_cv.fit(X_train_poly, y_train)

print(f"RidgeCV Results:")
print(f"  Optimal alpha: {ridge_cv.alpha_:.4f}")
print(f"  Test R²: {r2_score(y_test, ridge_cv.predict(X_test_poly)):.4f}")

# LassoCV: Automatic alpha selection using cross-validation
lasso_cv = LassoCV(alphas=alphas_cv, cv=5, max_iter=10000)
lasso_cv.fit(X_train_poly, y_train)

print(f"\nLassoCV Results:")
print(f"  Optimal alpha: {lasso_cv.alpha_:.4f}")
print(f"  Test R²: {r2_score(y_test, lasso_cv.predict(X_test_poly)):.4f}")
print(f"  Features selected: {np.sum(lasso_cv.coef_ != 0)}")

## 7. Visual Comparison: All Models

In [ ]:
# Compare all models visually
X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)
X_plot_poly = poly_features.transform(X_plot)

models = {
    'Linear (No Reg)': LinearRegression(),
    'Ridge': Ridge(alpha=ridge_cv.alpha_),
    'Lasso': Lasso(alpha=lasso_cv.alpha_, max_iter=10000),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

plt.figure(figsize=(15, 10))

for i, (name, model) in enumerate(models.items(), 1):
    model.fit(X_train_poly, y_train)
    y_plot = model.predict(X_plot_poly)
    y_test_pred = model.predict(X_test_poly)
    
    plt.subplot(2, 2, i)
    plt.scatter(X_train, y_train, alpha=0.6, s=50, edgecolors='k', label='Training')
    plt.scatter(X_test, y_test, alpha=0.8, s=70, c='red', edgecolors='k', label='Test')
    plt.plot(X_plot, y_plot, 'b-', linewidth=2, label='Model')
    plt.xlabel('X', fontsize=11)
    plt.ylabel('y', fontsize=11)
    plt.title(f'{name}\nTest R²={r2_score(y_test, y_test_pred):.3f}', fontsize=12)
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.ylim(-5, 12)

plt.tight_layout()
plt.show()

## 8. Real-World Example: Feature Selection with Lasso

In [ ]:
from sklearn.datasets import fetch_california_housing

# Load dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("California Housing Dataset:")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"  Feature names: {list(X.columns)}")

In [ ]:
# Compare models on real data
models_real = {
    'Linear': LinearRegression(),
    'Ridge': RidgeCV(alphas=np.logspace(-3, 3, 50)),
    'Lasso': LassoCV(alphas=np.logspace(-3, 3, 50), max_iter=10000),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

results_comparison = []
for name, model in models_real.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    if hasattr(model, 'coef_'):
        n_features = np.sum(np.abs(model.coef_) > 1e-5)
    else:
        n_features = X.shape[1]
    
    results_comparison.append({
        'Model': name,
        'Train R²': f"{r2_score(y_train, y_train_pred):.4f}",
        'Test R²': f"{r2_score(y_test, y_test_pred):.4f}",
        'Test RMSE': f"{np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}",
        'Features Used': n_features
    })

df_comparison = pd.DataFrame(results_comparison)
print("\nModel Comparison on California Housing:")
print(df_comparison.to_string(index=False))

In [ ]:
# Visualize feature importance for Lasso
lasso_model = models_real['Lasso']
feature_importance = pd.DataFrame({
    'Feature': housing.feature_names,
    'Coefficient': lasso_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['red' if c == 0 else 'blue' for c in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title(f'Lasso Feature Importance\n(Red = eliminated features)', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nFeature Selection by Lasso:")
print(feature_importance)

## Key Takeaways

### Ridge Regression (L2)
✅ **Pros:**
- Shrinks coefficients smoothly
- Works well when all features are relevant
- Closed-form solution (fast)
- Handles multicollinearity well

❌ **Cons:**
- Doesn't perform feature selection
- All features retained

### Lasso Regression (L1)
✅ **Pros:**
- Automatic feature selection (sparse solutions)
- Interpretable models
- Works well with irrelevant features

❌ **Cons:**
- Can arbitrarily select one feature from correlated group
- No closed-form solution (iterative)
- Can be unstable with highly correlated features

### Elastic Net
✅ **Pros:**
- Combines benefits of Ridge and Lasso
- Handles correlated features better than Lasso
- Encourages grouping effect

❌ **Cons:**
- Two hyperparameters to tune (alpha, l1_ratio)
- More complex

## When to Use Each

- **Ridge**: When all features are expected to be relevant, multicollinear features
- **Lasso**: When you suspect many features are irrelevant, need interpretability
- **Elastic Net**: When features are highly correlated and you want feature selection

## Next Steps

- **Cross-validation**: Systematic hyperparameter tuning
- **Feature engineering**: Create better features before regularization
- **Classification**: Logistic regression with regularization

## Exercises

1. Implement Ridge regression from scratch using gradient descent
2. Compare regularization performance on datasets with different numbers of features
3. Create a visualization showing the "regularization path" for all coefficients
4. Implement k-fold cross-validation manually to select optimal alpha
5. Explore how regularization affects model interpretability